# 05 — Visualization Prototyping & Storytelling

Prototype chart cho dashboard, mapping mỗi chart với Data Visualization concepts, và tạo storytelling cards.

In [ ]:

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)
PROJECT_ROOT = Path("..")
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "mobile_usage_behavioral_analysis.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
AGG_DIR = PROJECT_ROOT / "data" / "aggregated"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
for p in [PROCESSED_DIR, AGG_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
def title(t):
    print("\n" + "="*90); print(t); print("="*90)
def clean_cols(df):
    df = df.copy(); df.columns = [c.strip() for c in df.columns]; return df

title("Load final analytical dataset")
path = PROCESSED_DIR / "smartphone_clustered.csv"
if path.exists():
    df = pd.read_csv(path)
else:
    df = pd.read_csv(PROCESSED_DIR / "smartphone_features.csv")
print(df.shape)
display(df.head())


In [ ]:

title("Dashboard concept map")
concept_map = pd.DataFrame([
    ["Executive Overview", "KPI + Density Plot", "How intense is smartphone use overall?", "Distribution, preattentive highlight, redundant encoding, annotation"],
    ["Lifestyle Segmentation", "PCA Map + Cluster Profiles", "What digital lifestyle archetypes exist?", "PCA, clustering, color encoding, Gestalt enclosure"],
    ["Behavioral Relationships", "Scatter + Trendline + Density Contour", "What relates to screen time?", "Position on common scale, overplotting handling, uncertainty"],
    ["Demographic Explorer", "Heatmap + Small Multiples", "How do age, gender, location differ?", "Faceting, small multiples, heatmap, comparison"],
    ["Advanced Exploration", "Parallel Coordinates", "How do multiple variables interact?", "High-dimensional visualization, brushing, focus+context"],
    ["Storytelling", "Insight Cards", "What should the audience remember?", "Narrative flow, visual hierarchy, progressive disclosure"],
], columns=["Dashboard Section", "Chart", "Main Question", "DataViz Concepts"])
display(concept_map)
concept_map.to_csv(PROCESSED_DIR / "dashboard_concept_map.csv", index=False)


In [ ]:

title("Prototype 1 — Executive overview")
total = len(df)
heavy_users = df[df["Screen_Time_Segment"].isin(["Heavy (8–12h)", "Extreme (>12h)"])].shape[0]
heavy_rate = heavy_users / total * 100
kpis = pd.DataFrame({"KPI": ["Total Users", "Avg Screen Time", "Avg App Usage", "Avg Apps Used", "Heavy / Extreme Users"], "Value": [f"{total:,}", f"{df['Daily_Screen_Time_Hours'].mean():.2f}h", f"{df['Total_App_Usage_Hours'].mean():.2f}h", f"{df['Number_of_Apps_Used'].mean():.2f}", f"{heavy_rate:.1f}%"]})
display(kpis)
fig = px.histogram(df, x="Daily_Screen_Time_Hours", nbins=40, histnorm="probability density", color="Screen_Time_Segment", marginal="box", title="Daily screen time distribution — heavy users highlighted")
fig.add_vline(x=8, line_dash="dash", annotation_text="Heavy threshold")
fig.add_vline(x=12, line_dash="dash", annotation_text="Extreme threshold")
fig.show()


In [ ]:

title("Prototype 2 — Lifestyle segmentation")
if {"PCA_1", "PCA_2", "Cluster_Name"}.issubset(df.columns):
    fig = px.scatter(df, x="PCA_1", y="PCA_2", color="Cluster_Name", opacity=0.75, hover_data=["Age", "Gender", "Location", "Daily_Screen_Time_Hours"], title="PCA user map colored by lifestyle cluster")
    fig.update_layout(height=650)
    fig.show()
    counts = df["Cluster_Name"].value_counts().reset_index(); counts.columns = ["Cluster_Name", "Users"]; counts["Percentage"] = counts["Users"] / len(df) * 100
    display(counts)
    fig = px.bar(counts.sort_values("Users"), x="Users", y="Cluster_Name", orientation="h", text=counts.sort_values("Users")["Percentage"].round(1).astype(str)+"%", title="Lifestyle cluster size")
    fig.update_traces(textposition="outside")
    fig.show()
else:
    print("Run notebook 04 first for PCA and clusters.")


In [ ]:

title("Prototype 3 — Relationships and density")
fig = px.scatter(df, x="Number_of_Apps_Used", y="Daily_Screen_Time_Hours", color="Dominant_Lifestyle", opacity=0.55, trendline="ols", hover_data=["Age", "Gender", "Location", "Screen_Time_Segment"], title="Apps used vs daily screen time")
fig.show()
fig = px.density_contour(df, x="Number_of_Apps_Used", y="Daily_Screen_Time_Hours", title="Density contour — most common user region")
fig.update_traces(contours_coloring="fill", contours_showlabels=True)
fig.show()


In [ ]:

title("Prototype 4 — Demographics, heatmap, and small multiples")
age_order = ["18–24", "25–34", "35–44", "45–54", "55–59"]
age = df.groupby("Age_Group", as_index=False)[["Social_Media_Usage_Hours", "Productivity_App_Usage_Hours", "Gaming_App_Usage_Hours", "Total_App_Usage_Hours", "Daily_Screen_Time_Hours"]].mean().round(2)
age["Age_Group"] = pd.Categorical(age["Age_Group"], categories=age_order, ordered=True)
age = age.sort_values("Age_Group")
px.imshow(age.set_index("Age_Group"), text_auto=True, aspect="auto", title="Average usage hours by age group", labels=dict(color="Hours")).show()
px.histogram(df, x="Daily_Screen_Time_Hours", facet_col="Gender", color="Screen_Time_Segment", nbins=25, title="Small multiples — screen time by gender").show()
loc = df.groupby("Location", as_index=False)["Daily_Screen_Time_Hours"].mean().sort_values("Daily_Screen_Time_Hours")
fig = px.bar(loc, x="Daily_Screen_Time_Hours", y="Location", orientation="h", text="Daily_Screen_Time_Hours", title="Average screen time by location")
fig.update_traces(texttemplate="%{text:.2f}h", textposition="outside")
fig.show()


In [ ]:

title("Prototype 5 — Slopegraph-style comparison")
groups = {"Light Users (<4h)": df[df["Screen_Time_Segment"] == "Light (<4h)"], "Heavy Users (8h+)": df[df["Screen_Time_Segment"].isin(["Heavy (8–12h)", "Extreme (>12h)"])]}
metrics = ["Social_Media_Usage_Hours", "Productivity_App_Usage_Hours", "Gaming_App_Usage_Hours", "Total_App_Usage_Hours", "Number_of_Apps_Used"]
rows = []
for g, sub in groups.items():
    for m in metrics:
        rows.append({"Group": g, "Metric": m, "Average": sub[m].mean()})
slope = pd.DataFrame(rows)
display(slope.round(2))
fig = px.line(slope, x="Group", y="Average", color="Metric", markers=True, title="Slopegraph-style comparison — light vs heavy users")
fig.show()


In [ ]:

title("Storytelling cards export")
cards = pd.DataFrame([
    [1, "Nearly half are heavy or extreme users", f"{heavy_rate:.1f}% of users spend 8+ hours per day on screens.", "Hero KPI + highlighted density plot", "Preattentive pop-out, annotation, redundant encoding"],
    [2, "Digital lifestyles are not the same", "Users split into social, productivity and gaming-oriented behaviors.", "Cluster map + lifestyle profile cards", "PCA, clustering, Gestalt enclosure"],
    [3, "More apps does not fully explain more screen time", "Apps used and screen time show a relationship, but not a complete one.", "Scatter plot with trendline and density contour", "Position encoding, overplotting handling"],
    [4, "Location differences are more visible than gender differences", "City-level averages vary more clearly than gender-level averages.", "Sorted location bar + gender small multiples", "Length encoding, small multiples"],
    [5, "Digital lifestyle needs multiple lenses", "Screen time, app category, app diversity and demographics each reveal a different facet.", "Storytelling page with multiple charts", "Multiple charts, progressive disclosure, narrative flow"],
], columns=["Order", "Headline", "Evidence", "Recommended Visual", "Concept"])
display(cards)
cards.to_csv(PROCESSED_DIR / "storytelling_cards.csv", index=False)
print("Saved storytelling_cards.csv")
